# Lesson 2 — Prompt caching and the invalidation trap

**Module 2 · ~8 minutes · API key required**

Highest return-per-minute lesson in the course. Prompt caching is available on every major provider. Most teams leave it on the table — not because they forgot the flag, but because they put one dynamic token in the wrong place. Nothing errors. The bill just stays high.

> **Presenting:** show the raw usage object on screen. Engineers need the literal field names they will grep for tomorrow.

### What you will be able to do

1. Read cache metadata on **both** vendors (they report it differently — mixing them double-counts).
2. See a cache write, a cache hit, and a silent miss caused by a timestamp in the prefix.
3. Derive break-even (~2nd call) and the honest total-bill saving (~30%, not 90%).


### How to work through this notebook

Run cells **top to bottom**. Each section tells you what is about to happen *before* you run the code.

| Marker | What it means |
|---|---|
| **About to happen** | What the next cell will do |
| **Watch for** | The number or field that makes the point — pause on it |
| **Why it matters** | The Monday-morning decision this should change |
| **Presenting:** | Live-demo cue. Students: treat this as the takeaway |

A **cost ledger** prints at the end of every notebook that spends money.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
print(f"\nLive provider: {cfg.provider}")
print("Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.")


  Provider : none (offline)
  Arithmetic cells still run. Live cells will use rehearsal fallbacks.
  Add OPENAI_API_KEY or ANTHROPIC_API_KEY to .env for live calls.
  Rate card: verified 5 Sep 2026 — re-check before presenting.

Live provider: offline
Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.


The cell above loads `.env`, chooses **OpenAI or Anthropic** from the key you have, and prints the three model tiers this notebook will call.

**Watch for:** a banner with `Provider`, `floor`, `mid`, `frontier`.
- If it names a vendor, live cells will spend a few cents.
- If it says `offline`, arithmetic still runs. Live cells print a rehearsal fallback instead of crashing — useful on a plane, not a substitute for a real key on caching / routing / eval lessons.

Switch vendor by setting `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and re-running that cell.


---
## The stable prefix

**About to happen.** We build a ~2,000–3,000 token company-policy block. Caching only works above a minimum length (~1,024 tokens on most mid-tier models; Haiku is higher). Shorter prefixes silently do nothing — that looks like a cache bug and is actually a length bug.

**Watch for:** the token estimate and which usage fields this provider will show.

| Vendor | What you will grep | Gotcha |
|---|---|---|
| Anthropic | `cache_creation_input_tokens` vs `cache_read_input_tokens` | Separate buckets. A write is billed at ~1.25×; a read at ~0.1×. |
| OpenAI | `usage.prompt_tokens_details.cached_tokens` | Cached tokens are a **subset** of `prompt_tokens`. Add them again and you double-count. No write premium — the discount appears on the *next* call. |


In [2]:
import datetime, time

# Must exceed the model's minimum cacheable length (~1024 tokens for most
# current Sonnet/GPT mid-tiers; Haiku is higher). Shorter prefixes silently
# do nothing — that looks like a cache bug and is actually a length bug.
POLICY_BLOCK = (
    "You are a support assistant for Northwind Logistics.\n\n"
    + "\n".join(
        f"POLICY {i:03d}: Refund requests under $500 may be approved without escalation when the "
        f"shipment was delayed more than 48 hours and the customer has fewer than three prior claims "
        f"in the trailing twelve months. Document the decision under case note code RF-{i:03d}."
        for i in range(1, 60)
    )
)
print(f"policy block is ~{ntok(POLICY_BLOCK):,} tokens  (tiktoken estimate)")
print(f"provider: {cfg.provider}   model: {MODELS.mid}")
if cfg.provider == "anthropic":
    print("Watch cache_creation_input_tokens vs cache_read_input_tokens — disjoint buckets.")
else:
    print("Watch usage.prompt_tokens_details.cached_tokens — it is INSIDE prompt_tokens.")


policy block is ~3,019 tokens  (tiktoken estimate)
provider: offline   model: claude-sonnet-5
Watch usage.prompt_tokens_details.cached_tokens — it is INSIDE prompt_tokens.


---
## Call 1 — cold cache (you pay the write, or you miss)

**About to happen.** One question against the policy block, with caching requested. This is the first time this prefix has been seen.

**Watch for:**
- Anthropic: `cache_creation_input_tokens ≈` the policy size, `cache_read ≈ 0`. You paid the 1.25× write premium.
- OpenAI: `cached_tokens = 0`. Automatic cache; no write premium. The discount is on call 2.

**Why it matters.** The first call is supposed to look "expensive." That is the cache being built, not a failure.


In [3]:
r1 = complete(
    "What is the refund threshold?",
    system=POLICY_BLOCK,
    model=MODELS.mid,
    max_tokens=120,
    cache=True,
    label="1. cold cache (write)",
)
print("raw usage:")
print(getattr(r1.raw, "usage", r1.raw))


⚠ No API key — using a rehearsal result.
1. cold cache (write)                         $0.006848   in=3024    out=80     cw=0       cr=0       offline fallback
raw usage:
None


---
## Call 2 — warm cache, different question, identical prefix

**About to happen.** A *different* user question. The system prefix is byte-identical to call 1.

**Watch for:** `cache_read` (Anthropic) or `cached_tokens` (OpenAI) lighting up, and `cache_creation` going to 0. The printed cost should drop.

**Why it matters.** You do not need the same *question*. You need the same *prefix*. That is why caching is a prompt-ordering discipline: put the stable block first, the user turn last.

> **Presenting:** "The premium was recovered on the very next call."


In [4]:
time.sleep(1)
r2 = complete(
    "How many prior claims disqualify a customer?",
    system=POLICY_BLOCK,
    model=MODELS.mid,
    max_tokens=120,
    cache=True,
    label="2. warm cache (read)",
)
print("raw usage:")
print(getattr(r2.raw, "usage", r2.raw))
print()
if r2.cache_read:
    print("CACHE HIT. The premium (or the first-call miss) was recovered.")
else:
    print("No cache read. Prefix may be under the minimum, or the cache TTL expired.")


⚠ No API key — using a rehearsal result.
2. warm cache (read)                          $0.006856   in=3028    out=80     cw=0       cr=0       offline fallback
raw usage:
None

No cache read. Prefix may be under the minimum, or the cache TTL expired.


---
## The invalidation trap

**About to happen.** We inject `datetime.now()` *before* the policy block — the most common production mistake. Logging a session id, a request id, or `Date: ...` at the top of the system prompt does the same thing.

**Watch for:** `cache_read` collapsing to zero. `cache_creation` (Anthropic) returns. **No exception.** The API is happy. The bill quietly stays at full price, forever.

**Why it matters.** Caching is prefix matching. One changing byte anywhere in the prefix invalidates everything after it. Most teams "turned caching on" and then invalidated it with observability metadata.

> **Presenting:** let the zero sit on screen for a beat. Most rooms contain at least one team that will find this in their code this week.


In [5]:
BAD_PREFIX = f"Session started: {datetime.datetime.now().isoformat()}\n\n" + POLICY_BLOCK
r3 = complete(
    "What is the refund threshold?",
    system=BAD_PREFIX,
    model=MODELS.mid,
    max_tokens=120,
    cache=True,
    label="3. timestamp AT FRONT",
)
print()
print("^ cache_read collapsed. Full-price (or write-premium) on every single call.")
print("  Nothing raised an exception. The bill just quietly stayed high.")


⚠ No API key — using a rehearsal result.
3. timestamp AT FRONT                         $0.006888   in=3044    out=80     cw=0       cr=0       offline fallback

^ cache_read collapsed. Full-price (or write-premium) on every single call.
  Nothing raised an exception. The bill just quietly stayed high.


### The fix — move the dynamic content to the end

**About to happen.** Same timestamp, same policy, same question — but the timestamp now rides on the *user* message, after the cached prefix.

**Watch for:** the cache read coming back.

**The order that survives production:** system prompt → tool schemas → stable policy → retrieval → history → user turn. Static first. Dynamic last.


In [6]:
r4 = complete(
    f"What is the refund threshold?\n\n[session {datetime.datetime.now().isoformat()}]",
    system=POLICY_BLOCK,
    model=MODELS.mid,
    max_tokens=120,
    cache=True,
    label="4. timestamp AT END",
)
print()
print("^ static first, dynamic last. Same information, different position.")
print("Order: system prompt -> tool schemas -> stable policy -> retrieval -> history -> user turn.")


⚠ No API key — using a rehearsal result.
4. timestamp AT END                           $0.006886   in=3043    out=80     cw=0       cr=0       offline fallback

^ static first, dynamic last. Same information, different position.
Order: system prompt -> tool schemas -> stable policy -> retrieval -> history -> user turn.


\
---
## The break-even, derived from the rate card

**About to happen.** For each model, $$N_{\mathrm{breakeven}} = 1 + \frac{P_{\mathrm{write}} - P_{\mathrm{input}}}{P_{\mathrm{input}} - P_{\mathrm{read}}}$$

**Watch for:** every row landing near **call 2**. There is essentially no stable-prefix workload where caching loses money.

OpenAI's automatic cache has no write premium, so the "first miss, then hit" pattern is even more one-sided.


In [7]:
for m in ["claude-haiku-4-5", "claude-sonnet-5", "claude-opus-5", "claude-fable-5-1",
          "gpt-5.6-luna", "gpt-5.6-terra"]:
    n = cache_breakeven(m)
    p = PRICES[m]
    print(f"{m:<22} write={p['cw']:>6.2f}  read={p['cr']:>6.3f}  "
          f"break-even at call {n:.2f} -> call {int(-(-n // 1))}")
print("\nThere is essentially no stable-prefix workload where caching loses money.")


claude-haiku-4-5       write=  1.25  read= 0.100  break-even at call 1.28 -> call 2
claude-sonnet-5        write=  2.50  read= 0.200  break-even at call 1.28 -> call 2
claude-opus-5          write=  6.25  read= 0.500  break-even at call 1.28 -> call 2
claude-fable-5-1       write= 12.50  read= 0.250  break-even at call 1.26 -> call 2
gpt-5.6-luna           write=  0.25  read= 0.020  break-even at call 1.28 -> call 2
gpt-5.6-terra          write=  2.50  read= 0.200  break-even at call 1.28 -> call 2

There is essentially no stable-prefix workload where caching loses money.


---
## The honest caveat: 90% off input is NOT 90% off the bill

**About to happen.** We hold tokens-in / tokens-out fixed and sweep cache hit rate from 0% to 100%. Output is 5–6× the price of input and is **never** cached.

**Watch for:** the 80% hit-rate row. The "90% discount" lands as roughly **a third off the total bill**.

**Why it matters.** 90% is the input-only headline. ~30% is the honest planning number. Say it before someone in the room does the arithmetic themselves.

> **Presenting:** "At a realistic 80% hit rate, this is a 30-point lever, not a 90-point lever. Still the largest *easy* lever in the stack."


In [8]:
def bill(model, tok_in, tok_out, hit_rate):
    cached = tok_in * hit_rate
    return cost(model, inp=tok_in - cached, out=tok_out, cache_r=cached)

M, TIN, TOUT = MODELS.mid, 8000, 400
base = bill(M, TIN, TOUT, 0.0)
print(f"{'hit rate':>10} {'cost/call':>12} {'saving':>10}")
for h in [0, 0.25, 0.5, 0.8, 0.95, 1.0]:
    c = bill(M, TIN, TOUT, h)
    print(f"{h:>9.0%} {usd(c):>12} {1 - c / base:>9.0%}")
print()
print('At a realistic 80% hit rate the "90% discount" lands as roughly a third off the total.')
print("That ~30% is the honest planning number. 90% is the input-only headline.")


  hit rate    cost/call     saving
       0%      $0.0200        0%
      25%      $0.0164       18%
      50%      $0.0128       36%
      80%    $0.008480       58%
      95%    $0.006320       68%
     100%    $0.005600       72%

At a realistic 80% hit rate the "90% discount" lands as roughly a third off the total.
That ~30% is the honest planning number. 90% is the input-only headline.


In [9]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.0275


,label,model,input,output,cache_write,cache_read,usd,note
0,1. cold cache (write),claude-sonnet-5,3024,80,0,0,0.006848,offline fallback
1,2. warm cache (read),claude-sonnet-5,3028,80,0,0,0.006856,offline fallback
2,3. timestamp AT FRONT,claude-sonnet-5,3044,80,0,0,0.006888,offline fallback
3,4. timestamp AT END,claude-sonnet-5,3043,80,0,0,0.006886,offline fallback


---
## Takeaways

- Caching is a **prompt-ordering discipline**, not a feature flag.
- Static first. Dynamic last. One timestamp in the wrong place costs **12.5×** more than a hit (1.25× write vs 0.1× read on Anthropic).
- Watch `cache_read_input_tokens` (Anthropic) or `cached_tokens` (OpenAI) in production. Near zero means you have this bug.
- **Normalisation:** OpenAI counts cached tokens *inside* `prompt_tokens`. Anthropic reports them in a *separate* bucket. Sum naively and you double-count.
- Plan for **~30% off the total bill** at a healthy hit rate, not 90%.

**Try on Monday:** log the cache-read field on 100 production calls. If the hit rate is near zero, grep the system prompt for timestamps, request ids, and anything else that changes per call.
